In [1]:
import ipytest
import pytest
import pandas as pd
import urllib.request
from datetime import date
import json

ipytest.autoconfig()

In [3]:
def import_json_to_dict(url) :
    response = urllib.request.urlopen(url)
    my_dict = json.loads(response.read())
    return my_dict


taxo_dep_url = 'https://raw.githubusercontent.com/etalab/dashboard-aides-entreprises/master/backend/json/taxonomies/departements-minify.json'
taxo_reg_url = 'https://raw.githubusercontent.com/etalab/dashboard-aides-entreprises/master/backend/json/taxonomies/regions-minify.json'

taxo_dep_df = pd.DataFrame(import_json_to_dict(taxo_dep_url))
dep_list = list(taxo_dep_df['dep'])
print('{} departements.'.format(len(dep_list)))

taxo_reg_df = pd.DataFrame(import_json_to_dict(taxo_reg_url))
reg_list = list(taxo_reg_df['reg'])
print('{} regions.'.format(len(reg_list)))

pp_dep = pd.read_csv('../pp_dep.csv', sep=';', dtype={"reg":str, "dep":str}, parse_dates=['period_date'])

101 departements.
18 regions.


In [4]:
#%%run_pytest[clean]

def test_got_all_departments():
    # Vérifier si tous les départements sont présents dans le fichier
    assert sorted(pp_dep['dep'].unique()) == sorted(taxo_dep_df['dep'])


def test_got_all_regions():
    # Vérifier si tous les régions sont présents dans le fichier
    assert sorted(pp_dep['reg'].unique()) == sorted(taxo_dep_df['reg'].unique())
    

def test_actual_value():
    # Vérfier si on ne rencontre pas de date future (eg données Objectif...)
    today = date.today().strftime("%Y-%m")
    assert (pp_dep['period_date'].dt.strftime('%Y-%m') > today).sum() == 0
    


In [5]:
ipytest.run('-qq')

...                                                                      [100%]


In [8]:
ref_indic = pd.read_csv('../refs/ref_indicateurs.csv', sep=';')

In [10]:
ref_indic

,indicateur,code_mesure,mesure
0,Nombre de repas servis dans les restaurants un...,RRU,Repas à 1 € dans les restaurants universitaires
1,Nombre d’entrées en service civique - SCI,SCI,Service civique
2,Nombre de PME - BPI,BPI,Modernisation de la filière (BPI)
3,Taux d'agents équipés pour le travail à distan...,PTA,Nouveau poste de travail de l’agent public
4,"Nombre d’exploitations certifiées ""haute valeu...",CIC,Crédit d’impôt pour la certification HVE
...,...,...,...
174,Nombre d’entreprises bénéficiares - APS,APS,Soutien par appels à projet au secteur spatial
175,Nombre de documents envoyés dans le dossier mé...,SNU,Ségur numérique
176,Nombre de projets incluant une transformation ...,FAA,Soutien aux fonds propres des filières aéro et...
177,Montant total des projets financés - PIA,PIA,Financement et valorisation de la recherche pa...


In [105]:
ALL_DEPS = sorted(taxo_dep_df['dep'].unique())
ALL_INDICS = ref_indic['indicateur'].unique()
MESURES = sorted(ref_indic['mesure'].unique())


def print_indic_count():
    """Affiche le nombre d'indicateurs récupérés par rapport au nombre d'indicateurs dans la table référence."""
    num_total_indic = ALL_INDICS.shape[0]
    num_caught_indic = pp_dep['indicateur'].unique().shape[0]
    print(f"Nombre d'indicateurs récupérés : {num_caught_indic}/{num_total_indic}")
    
    
def print_mesure_count():
    """Affiche le nombre de mesures récupérées par rapport au nombre de mesures inscrit dans la table de référence des indicateurs."""
    num_total_mesure = ref_indic['mesure'].unique().shape[0]
    num_caught_mesure = pp_dep['mesure'].unique().shape[0]
    print(f"Nombre de mesures récupérées : {num_caught_mesure}/{num_total_mesure}")
    
    
def print_indic_count_for_department(dep):
    """Affiche le nombre d'indicateurs récupérés pour le département donné."""
    df_dep = pp_dep[pp_dep['dep'] == dep]
    num_total_indic = ALL_INDICS.shape[0]
    num_caught_indic_dep = df_dep['indicateur'].shape[0]
    print(f"----Nombre d'indicateurs récupérés : {num_caught_indic_dep} / {num_total_indic}")

    
def print_mesure_count_for_department(dep):
    """Affiche le nombre de mesure récupérées pour le département donné."""
    df_dep = pp_dep[pp_dep['dep'] == dep]
    num_total_indic = ref_indic['mesure'].unique().shape[0]
    num_caught_mesure_dep = df_dep['mesure'].unique().shape[0]
    print(f"----Nombre de mesures récupérées : {num_caught_mesure_dep} / {num_total_indic}")

    
def print_counts_per_departments():
    """Affiche les mesures de comptage pour chaque département."""
    num_total_indic = ref_indic.shape[0]
    for dep in ALL_DEPS:
        print(f"Département {dep} :")
        print_indic_count_for_department(dep)
        print_mesure_count_for_department(dep)
        print()


def print_counts_dep_per_mesure_indic():
    """Affiche les mesures de comptage pour chaque indicateur (regroupé par mesure)."""
    for mesure in MESURES:
        print(f"Mesure : {mesure}")
        indics = ref_indic[ref_indic['mesure'] == mesure]['indicateur'].unique()
        for indic in indics:
            print(f'----Indicateur : {indic}')
            df_indic = pp_dep[pp_dep['indicateur'] == indic]
            num_deps = df_indic['dep'].unique().shape[0]
            print(f"--------Nombre de départements : {num_deps} / {len(ALL_DEPS)}")
            print()
        


In [106]:
print_indic_count()
print_mesure_count()
print_counts_dep_per_mesure_indic()
print_counts_per_departments()

Nombre d'indicateurs récupérés : 60/179
Nombre de mesures récupérées : 36/104
Mesure : Abondement de fonds régionaux
----Indicateur : Montant des investissements Etat dans les fonds régionaux - AFG
--------Nombre de départements : 0 / 101

----Indicateur : Montant total des investissements ainsi déclenchés (effet de levier) - AFG
--------Nombre de départements : 0 / 101

----Indicateur : Nombre de TPE/PME/ETI financées - AFG
--------Nombre de départements : 0 / 101

Mesure : Activité partielle de longue durée
----Indicateur : Nombre de salariés en activité partielle de longue durée - APL
--------Nombre de départements : 0 / 101

Mesure : Aide aux investissements de protection face aux aléas climatiques
----Indicateur : Surface agricole utile (SAU) couverte par des investissements de lutte contre les aléas climatiques (ha) - IEP
--------Nombre de départements : 0 / 101

Mesure : Aide à la relance de la construction durable (maires densificateurs)
----Indicateur : Nombre de communes béné